# BirdCLEF+ 2026 — EfficientNet v1 Submission

Self-contained inference notebook.

## Input
- `birdclef-2026` (Competition)
- Training notebook output (`fold0_best.pth` + `label_encoder.pkl`)

## Settings
| Setting | Value |
|---------|-------|
| Accelerator | None (CPU) |
| Internet | OFF |

In [ ]:
import os, glob, collections, time, pickle, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print('Device: CPU')

In [ ]:
# ============================================================
# CONFIG + PATH DETECTION
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p):
        COMP_DATA = p
        break
assert COMP_DATA, 'Competition data not found'

# Find model files from training notebook output
PTH_PATH = None
for f in glob.glob('/kaggle/input/**/*.pth', recursive=True):
    PTH_PATH = f
    break
assert PTH_PATH, 'fold0_best.pth not found. Add training notebook output as Input.'

LE_PATH = None
for f in glob.glob('/kaggle/input/**/label_encoder.pkl', recursive=True):
    LE_PATH = f
    break
assert LE_PATH, 'label_encoder.pkl not found.'

# Must match training config exactly
MODEL_NAME = 'tf_efficientnet_b3_ns'
USE_GEM = True
GEM_P = 3.0
SAMPLE_RATE = 32000
DURATION = 5
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 50
FMAX = 16000
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
WINDOW_SAMPLES = SAMPLE_RATE * DURATION
BATCH_SIZE = 32
OUTPUT_DIR = '/kaggle/working'

print(f'Competition data: {COMP_DATA}')
print(f'Model: {PTH_PATH}')
print(f'LabelEncoder: {LE_PATH}')

In [ ]:
# ============================================================
# MODEL + AUDIO UTILS (all definitions)
# ============================================================
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p).flatten(1)

class BirdCLEFModel(nn.Module):
    def __init__(self, model_name, num_classes, use_gem=True, gem_p=3.0):
        super().__init__()
        self.use_gem = use_gem
        if use_gem:
            self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0, global_pool='', in_chans=3)
            self.pool = GeMPooling(p=gem_p)
        else:
            self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0, in_chans=3)
            self.pool = None
        self.head = nn.Linear(self.backbone.num_features, num_classes)
    def forward(self, x):
        if self.use_gem:
            return self.head(self.pool(self.backbone.forward_features(x)))
        return self.head(self.backbone(x))
    @torch.no_grad()
    def predict(self, x):
        return torch.sigmoid(self.forward(x))

def audio_to_melspec(waveform):
    mel = librosa.feature.melspectrogram(y=waveform, sr=SAMPLE_RATE, n_mels=N_MELS,
        n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mn, mx = mel_db.min(), mel_db.max()
    if mx - mn > 1e-6:
        return ((mel_db - mn) / (mx - mn)).astype(np.float32)
    return np.zeros_like(mel_db, dtype=np.float32)

def window_to_tensor(window):
    mel = audio_to_melspec(window)
    t = torch.from_numpy(mel).unsqueeze(0).repeat(3, 1, 1)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t - mean) / std

print('Definitions OK')

In [ ]:
# ============================================================
# LOAD MODEL + LABEL ENCODER
# ============================================================
device = torch.device('cpu')

with open(LE_PATH, 'rb') as f:
    le = pickle.load(f)
num_classes = len(le.classes_)
print(f'Classes: {num_classes}')

model = BirdCLEFModel(MODEL_NAME, num_classes, USE_GEM, GEM_P)
ckpt = torch.load(PTH_PATH, map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded {os.path.basename(PTH_PATH)} (val_cmap={ckpt.get("val_score", "?")})')

sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
species_to_idx = {}
for sp in species_cols:
    try:
        species_to_idx[sp] = int(le.transform([sp])[0])
    except ValueError:
        pass
print(f'Mapped: {len(species_to_idx)} / {len(species_cols)} species')

In [ ]:
# ============================================================
# INFERENCE
# ============================================================
test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')

@torch.no_grad()
def predict_batch(tensors):
    batch = torch.stack(tensors, dim=0)
    return model.predict(batch).numpy()

bucket_preds = collections.defaultdict(list)
t0 = time.time()

for fi, fname in enumerate(test_files):
    path = os.path.join(test_dir, fname)
    stem = os.path.splitext(fname)[0]
    waveform, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(waveform) == 0:
        continue
    
    windows, keys = [], []
    for start in range(0, len(waveform), WINDOW_SAMPLES):
        window = waveform[start:start + WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        bucket_sec = (start // WINDOW_SAMPLES) * DURATION
        windows.append(window_to_tensor(window))
        keys.append(f'{stem}_{bucket_sec}')
        if len(windows) >= BATCH_SIZE:
            probs = predict_batch(windows)
            for k, p in zip(keys, probs):
                bucket_preds[k].append(p)
            windows, keys = [], []
    if windows:
        probs = predict_batch(windows)
        for k, p in zip(keys, probs):
            bucket_preds[k].append(p)
    
    if (fi + 1) % 50 == 0:
        print(f'  [{fi+1}/{len(test_files)}] elapsed={int(time.time()-t0)}s', flush=True)

print(f'Inference done: {int(time.time()-t0)}s, buckets={len(bucket_preds)}')

In [ ]:
# ============================================================
# BUILD SUBMISSION
# ============================================================
rows = []
for row_id, pred_list in bucket_preds.items():
    avg = np.mean(pred_list, axis=0)
    row = {'row_id': row_id}
    for sp in species_cols:
        row[sp] = float(avg[species_to_idx[sp]]) if sp in species_to_idx else 0.0
    rows.append(row)

submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)
submission = submission.set_index('row_id').reindex(sample_sub['row_id'], fill_value=0.0).reset_index()

out_path = f'{OUTPUT_DIR}/submission.csv'
submission.to_csv(out_path, index=False)

assert len(submission) == len(sample_sub), 'Row count mismatch'
assert list(submission.columns) == list(sample_sub.columns), 'Column mismatch'
assert submission.isnull().sum().sum() == 0, 'NaN found'

vals = submission.drop('row_id', axis=1).values
print(f'Submission: {out_path}')
print(f'  Rows: {len(submission)}')
print(f'  Mean: {vals.mean():.6f}')
print(f'  Max: {vals.max():.4f}')
print(f'  Zero: {(vals == 0).mean():.2%}')